# FraudMind Milestone 5A — Sentinel v2 Demo

**Bounded ReAct investigator.** Sentinel now uses `create_react_agent` with a maximum of 3 tool calls per investigation.

| Tool | Purpose |
|------|---------|
| `get_enrichment` | Account age, chargebacks, step-up history, device reuse, linked entities |
| `find_similar_cases` | Memory search by pattern tags |
| `get_specialist_evidence` | Score + primary signals for a specific domain |

This notebook runs Case 1 (ring attack) and shows the tool trace: which tools were called, in what order, and what they returned.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.fraud_graph import fraud_graph
from src.schemas.ato_schemas import MockATOSignals
from src.schemas.payment_schemas import PaymentSignals
from src.schemas.identity_schemas import IdentitySignals
from src.schemas.promo_schemas import PromoSignals
from src.schemas.ring_schemas import RingSignals
from src.schemas.payload_schemas import PayloadSignals
from src.sentinel.investigation_agent import run_investigation

## Case 1: Ring Attack

Full ring attack. All 6 specialists fire. Known ring signature match forces a `block` verdict.
Sentinel triggers because `aggregate >= 0.85` with `specialists_run >= 3`.

In [ ]:
case1_state = {
    "primary_entity_id": "acc_ring_001",
    "primary_entity_type": "user",
    "event_type": "authentication",

    "ato_signals": MockATOSignals(
        account_id="acc_ring_001",
        account_age_days=14,
        is_new_device=True,
        device_id="dev_shared_factory_01",
        login_geo="Lagos, Nigeria",
        account_home_geo="San Jose, CA",
        geo_mismatch=True,
        time_since_last_login_days=1,
        immediate_high_value_action=True,
        email_change_attempted=True,
        password_change_attempted=False,
        support_ticket_language_match=False,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_ring_001",
        account_id="acc_ring_001",
        amount_usd=1800.00,
        merchant_category="electronics",
        merchant_country="Nigeria",
        account_home_country="United States",
        is_international=True,
        transactions_last_1h=7,
        transactions_last_24h=12,
        avg_transaction_amount_30d=75.00,
        amount_vs_avg_ratio=24.0,
        is_first_transaction=False,
        card_present=False,
        days_since_account_creation=14,
    ),
    "identity_signals": IdentitySignals(
        account_id="acc_ring_001",
        account_age_days=14,
        email_domain="tempmail.io",
        is_disposable_email=True,
        phone_country_matches_ip_country=False,
        address_provided=True,
        address_is_commercial=True,
        document_verification_passed=False,
        pii_seen_on_other_accounts=6,
        accounts_created_from_same_device=22,
        accounts_created_from_same_ip=15,
        signup_velocity_same_ip_24h=8,
        name_dob_mismatch=True,
    ),
    "promo_signals": PromoSignals(
        account_id="acc_ring_001",
        promo_code="REF-RING-001",
        promo_type="referral",
        account_age_days=14,
        is_first_order=True,
        promo_uses_this_account=3,
        promo_uses_same_code=1,
        accounts_linked_same_device=10,
        accounts_linked_same_ip=12,
        referrer_account_id="acc_ring_ref_001",
        referrer_account_age_days=7,
        device_shared_with_referrer=True,
        email_domain_pattern_suspicious=True,
        order_cancelled_after_promo=True,
        payout_requested_immediately=True,
    ),
    "ring_signals": RingSignals(
        primary_entity_id="acc_ring_001",
        primary_entity_type="user",
        device_id="dev_shared_factory_01",
        ip_address_hash="hash_known_ring_ip_001",
        accounts_sharing_device=22,
        accounts_sharing_ip=15,
        linked_account_ids=["acc_ring_002", "acc_ring_003", "acc_ring_004"],
        linked_accounts_with_block_verdict=3,
        linked_accounts_with_fraud_confirmed=2,
        shared_payment_method_across_accounts=True,
        payment_method_account_count=7,
        known_ring_signature_match=True,
        ring_signature_id="ring_WEST_AFRICA_001",
        velocity_account_creation_same_ip_7d=18,
        transaction_graph_min_hops_to_fraud=1,
    ),
    "payload_signals": PayloadSignals(
        session_id="sess_ring_001",
        account_id="acc_ring_001",
        user_agent_string="HeadlessChrome/118.0",
        user_agent_is_headless=True,
        tls_fingerprint_ja3="a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4",
        tls_fingerprint_known_bot=True,
        http_header_order_anomaly=True,
        accept_language_missing=True,
        mouse_movement_entropy=0.0,
        keystroke_dynamics_score=0.04,
        requests_per_minute=210.0,
        request_timing_variance_ms=2.1,
        captcha_solve_time_ms=85,
        captcha_solve_pattern_automated=True,
        credential_stuffing_ip_on_blocklist=True,
        login_attempt_count_this_session=9,
        api_endpoint_sequence_anomaly=True,
    ),
}

# Run through execution system first
result1 = fraud_graph.invoke(case1_state)
arbiter_output = result1["arbiter_output"]
risk_vector = result1["risk_vector"]

print(f"Arbiter verdict    : {arbiter_output.verdict}")
print(f"Arbiter confidence : {arbiter_output.confidence:.4f}")
print(f"Trigger async      : {arbiter_output.trigger_async}")
print(f"Trigger reason     : {arbiter_output.trigger_reason}")
print(f"Risk aggregate     : {risk_vector.aggregate:.4f}")
print(f"Dominant domain    : {risk_vector.dominant_domain} ({risk_vector.dominant_score:.4f})")

## Run Sentinel v2

The ReAct agent decides which tools to call. For a ring attack the dominant domain is `ring`, so we expect Sentinel to start with `get_specialist_evidence("ring")` or `get_enrichment` depending on what it judges most useful first.

In [ ]:
specialist_scores = {
    "ato":      result1.get("ato_score"),
    "payment":  result1.get("payment_score"),
    "identity": result1.get("identity_score"),
    "promo":    result1.get("promo_score"),
    "ring":     result1.get("ring_score"),
    "payload":  result1.get("payload_score"),
}

record = run_investigation(
    entity_id="acc_ring_001",
    arbiter_output=arbiter_output,
    risk_vector=risk_vector,
    specialist_scores=specialist_scores,
)

print(f"Case ID   : {record.case_id}")
print(f"Assessment: {record.assessment}")
print(f"Tags      : {record.pattern_tags}")
print()
print("Narrative:")
print(record.investigation_narrative)

## Tool Trace

Which tools did Sentinel call, in what order, and what did each return?

In [ ]:
print(f"Total tool calls: {len(record.tool_trace)} / 3 max\n")
for entry in record.tool_trace:
    print(f"  [{entry['call_order']}] {entry['tool_name']}")
    print(f"      args    : {entry['arguments']}")
    print(f"      result  : {entry['result_summary']}")
    print()